# Tai hien pipeline WHU-LX XGB-DQN

Notebook nay duoc viet lai tu `scripts/reproduce_whulx_pipeline.py`. Muc tieu la tach pipeline thanh tung buoc ro rang de de giai thich, chay lai va dua ket qua vao bao cao tien do.

Pipeline gom 5 phan chinh:

1. Doc va lam sach dataset WHU-LX.
2. Huan luyen XGBoost de du doan `Differ_Indoor_Temp`.
3. Dung XGBoost nhu moi truong chuyen trang thai gan dung.
4. Huan luyen DQN de chon action HVAC/cua so.
5. Danh gia chinh sach DQN so voi human baseline va cac baseline co dinh.

## 1. Chuan bi moi truong

Cell nay xac dinh thu muc goc cua project, them `scripts/` vao Python path va import cac ham da tach san trong file script. Cach lam nay giu notebook gon, dong thoi van dam bao logic dung voi pipeline chinh.

In [ ]:
from pathlib import Path
import json
import random
import sys

import numpy as np
import pandas as pd
import tensorflow as tf

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from reproduce_whulx_pipeline import (
    DQN,
    ReplayBuffer,
    choose_day,
    evaluate_many_days,
    load_and_prepare_data,
    rollout_day,
    summarize_day,
    summarize_human,
    train_xgboost,
)

np.random.seed(2022)
random.seed(2022)
tf.random.set_seed(2022)

PROJECT_ROOT

## 2. Cau hinh duong dan va tham so

`repo_dir` tro toi thu muc `data/` trong repo, noi co `Cleaned_data.csv` lay tu WHU-LX. `out_dir` la noi luu cac file ket qua de bao cao DOCX co the doc lai tu `result.json`, `summary_metrics.csv` va `training_history.csv`.

Mac dinh script dung 1000 episodes de bam sat notebook goc. Khi muon thu nhanh, co the giam `EPISODES`, nhung so lieu bao cao nen dung lai cau hinh 1000 episodes.

In [ ]:
REPO_DIR = PROJECT_ROOT / "data"
OUT_DIR = PROJECT_ROOT / "artifacts" / "outputs" / "whulx_reproduction"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPISODES = 1000
DAY_INDEX = 0
XGB_DEVICE = "cpu"

print("Repo du lieu:", REPO_DIR)
print("Thu muc output:", OUT_DIR)

## 3. Doc va lam sach du lieu

Ham `load_and_prepare_data()` thuc hien cac buoc:

- Doc `Cleaned_data.csv` voi encoding `gbk`.
- Chuyen `Date_Time` sang kieu datetime.
- Loai bo missing value va gia tri sentinel `-999`.
- Ma hoa cac cot object bang `LabelEncoder` de mo hinh co the xu ly dang so.

Sau buoc nay, `raw` la du lieu goc va `data` la ban da lam sach/ma hoa.

In [ ]:
raw, data = load_and_prepare_data(REPO_DIR)

print("Raw shape:", raw.shape)
print("After clean shape:", data.shape)
data.head()

## 4. Huan luyen XGBoost transition model

XGBoost duoc dung de du doan `Differ_Indoor_Temp`, tuc do thay doi nhiet do trong nha o buoc ke tiep. Khi DQN thu mot action, pipeline dua action vao bang feature, goi XGBoost de uoc luong nhiet do tiep theo, sau do tinh reward.

Cac cot muc tieu truc tiep va cot dinh danh/thoi gian duoc loai khoi input. Ket qua can theo doi gom MAE, RMSE va R2.

In [ ]:
model_xgb, xgb_metrics = train_xgboost(data, device=XGB_DEVICE)
xgb_metrics

## 5. Chon mot ngay mau de rollout DQN

Notebook goc rollout tren mot ngay mau. Ham `choose_day()` cat 23 dong bat dau tu `DAY_INDEX * 24`, sau do tao hai bang:

- `data_test`: cac bien state ma DQN quan sat.
- `xgboost_test`: tap feature day du hon de XGBoost du doan transition.

Khung dieu khien chinh la cac step 6 den 17, tuong ung 12 buoc dieu khien trong ngay.

In [ ]:
data_test, xgboost_test = choose_day(DAY_INDEX * 24, data)

print("data_test:", data_test.shape)
print("xgboost_test:", xgboost_test.shape)
data_test.head()

## 6. Khoi tao DQN va replay buffer

DQN nhan state gom 8 feature dau cua `data_test` va tra ve Q-value cho 24 action. Action duoc ma hoa nhu sau:

- `0`: tat AC, dong cua so.
- `1-11`: bat AC, dong cua so, target temperature tu 20 den 30 C.
- `12`: tat AC, mo cua so.
- `13-23`: bat AC va mo cua so, target temperature tu 20 den 30 C.

Reward phat sai lech khoi vung tien nghi ASHRAE va phat them chi phi khi dung AC.

In [ ]:
num_actions = 24
q_network = DQN(num_actions)
q_network(np.zeros((1, 8), dtype=np.float32))

replay_buffer = ReplayBuffer(10000)
optimizer = tf.optimizers.Adam(0.001)
loss_fn = tf.losses.MeanSquaredError()

epsilon = 1.0
min_epsilon = 0.1
epsilon_decay = 0.995

train_cfg = {
    "num_actions": num_actions,
    "batch_size": 32,
    "gamma": 0.9,
    "optimizer": optimizer,
    "loss_fn": loss_fn,
}

## 7. Huan luyen DQN

Moi episode rollout lai ngay mau. Trong moi buoc dieu khien, agent chon action bang epsilon-greedy, XGBoost du doan nhiet do tiep theo, reward duoc tinh va transition duoc dua vao replay buffer.

Khi buffer du batch size, mang Q duoc cap nhat bang loss MSE giua Q hien tai va target Q. Epsilon giam dan de chuyen tu kham pha sang khai thac.

In [ ]:
history = []

for episode in range(EPISODES):
    _, actions, rewards, losses = rollout_day(
        model_xgb,
        q_network,
        data_test,
        xgboost_test,
        epsilon,
        train=True,
        replay_buffer=replay_buffer,
        train_cfg=train_cfg,
    )
    if epsilon > min_epsilon:
        epsilon *= epsilon_decay
    history.append(
        {
            "episode": episode + 1,
            "reward": float(np.sum(rewards)),
            "epsilon": float(epsilon),
            "avg_loss": float(np.mean(losses)) if losses else np.nan,
        }
    )
    if (episode + 1) % 100 == 0:
        print(f"Episode {episode + 1}/{EPISODES}, reward={np.sum(rewards):.2f}, epsilon={epsilon:.3f}")

history_df = pd.DataFrame(history)
history_df.tail()

## 8. Danh gia tren ngay mau

Sau khi train, agent duoc rollout voi `epsilon=0.0`, tuc la chon action co Q-value cao nhat. Phan nay so sanh DQN voi human baseline cua ngay mau ve comfort, nhiet do trung binh, ti le bat AC va ti le mo cua so.

In [ ]:
eval_day, eval_actions, eval_rewards, _ = rollout_day(
    model_xgb,
    q_network,
    data_test,
    xgboost_test,
    epsilon=0.0,
    train=False,
    train_cfg={"num_actions": num_actions},
)

human_metrics = summarize_human(data_test)
dqn_metrics = summarize_day(eval_day, eval_actions)

action_counts = pd.Series(eval_actions).value_counts().sort_index()
action_distribution = {
    int(action): float(count / len(eval_actions) * 100)
    for action, count in action_counts.items()
}

single_day_summary = pd.DataFrame(
    [
        {"controller": "Human", **human_metrics},
        {
            "controller": "DQN",
            **dqn_metrics,
            "total_reward": float(np.sum(eval_rewards)),
            "actions": eval_actions,
            "action_distribution_pct": action_distribution,
        },
    ]
)
single_day_summary

## 9. Danh gia mo rong tren nhieu ngay

De tranh ket luan dua tren mot ngay duy nhat, script danh gia them tat ca cac ngay hoan chinh trong dataset. Cac controller duoc so sanh gom:

- `DQN`: chinh sach hoc duoc.
- `Human`: hanh vi nguoi dung trong du lieu.
- `Off_Closed`: tat AC va dong cua.
- `Window_Open`: tat AC va mo cua.
- `AC_25_Closed`, `AC_26_Closed`, `AC_27_Closed`: baseline dat AC co dinh.

In [ ]:
max_complete_days = max(1, (len(data) - 24) // 24)
eval_indices = list(range(max_complete_days))

evaluation_summary, evaluation_action_distribution = evaluate_many_days(
    model_xgb,
    q_network,
    data,
    eval_indices,
    num_actions,
)

evaluation_df = pd.DataFrame(evaluation_summary)
evaluation_df

## 10. Luu artifact de cap nhat bao cao

Cell cuoi ghi lai 3 file quan trong:

- `training_history.csv`: reward/loss theo episode.
- `summary_metrics.csv`: bang so sanh cac controller tren nhieu ngay.
- `result.json`: toan bo metric de script tao DOCX chen vao bao cao.

In [ ]:
result = {
    "episodes": EPISODES,
    "day_index": DAY_INDEX,
    "raw_shape": list(raw.shape),
    "after_clean_shape": list(data.shape),
    "xgboost": xgb_metrics,
    "human_baseline": human_metrics,
    "dqn": {
        **dqn_metrics,
        "total_reward": float(np.sum(eval_rewards)),
        "actions": [int(a) for a in eval_actions],
        "action_distribution_pct": action_distribution,
    },
    "reported_whulx_readme": {
        "comfort_duration_increase_pct": 24.0,
        "ac_usage_decrease_pct": 24.7,
    },
    "evaluation": {
        "eval_days": max_complete_days,
        "controllers": evaluation_summary,
        "action_distribution_pct": evaluation_action_distribution,
    },
}

history_df.to_csv(OUT_DIR / "training_history.csv", index=False)
evaluation_df.to_csv(OUT_DIR / "summary_metrics.csv", index=False)
(OUT_DIR / "result.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
q_network.save_weights(OUT_DIR / "whulx_dqn_reproduction.weights.h5")

print(json.dumps(result, indent=2))

## 11. Dien giai ket qua moi nhat

Voi file `result.json` hien co trong workspace, pipeline da cho cac ket qua chinh sau:

- Du lieu goc: 42,934 dong x 27 cot; sau lam sach: 42,338 dong x 27 cot.
- XGBoost: MAE = 0.1823 C, RMSE = 0.3482 C, R2 = 0.4797.
- Ngay mau dau tien: human baseline va DQN deu dat 100% comfort trong khung 6-17h.
- DQN ngay mau khong bat AC, mo cua so 41.67% so buoc dieu khien.
- Danh gia 1,763 ngay: DQN dat 67.99% comfort, AC on 0.00%, window open 34.09%.

Nhan xet quan trong: DQN da chay lai duoc pipeline, nhung reward hien tai phat chi phi AC manh nen agent thien ve hanh dong khong dung dieu hoa. Vi vay ket qua nay phu hop de xac nhan logic XGB-DQN, con neu muon toi uu comfort tot hon can tiep tuc can chinh reward/action space.